In [ ]:
! pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121

In [ ]:
! pip install yt_dlp moviepy==1.0.3 flash_attn==2.7.3 Pillow==10.1.0 torch==2.3.1 torchaudio==2.3.1 torchvision==0.18.1 transformers==4.44.2 librosa==0.9.0 soundfile==0.12.1 vector-quantize-pytorch==1.18.5 vocos==0.1.0 decord moviepy

### ***Download sample video***
- Fill "url" first and run
- Then it will be downloaded as "test.mp4"

In [ ]:
import yt_dlp

url = ""  # describe video url here

ydl_opts = {
    # "outtmpl": "%(title)s.%(ext)s",
    "outtmpl": "test.mp4",
    "format": "mp4",
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([url])

## ***Laading MiniCPM-o-2_6 model***
- Using code snippet introduced in huggingface
- To reproduce, we need to install exactly same modulea above

In [ ]:
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer

# load omni model default, the default init_vision/init_audio/init_tts is True
# if load vision-only model, please set init_audio=False and init_tts=False
# if load audio-only model, please set init_vision=False
model = AutoModel.from_pretrained(
    'openbmb/MiniCPM-o-2_6',
    trust_remote_code=True,
    attn_implementation='sdpa', # sdpa or flash_attention_2
    torch_dtype=torch.bfloat16,
    init_vision=True,
    init_audio=True,
    init_tts=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.eval().cuda()
tokenizer = AutoTokenizer.from_pretrained('openbmb/MiniCPM-o-2_6', trust_remote_code=True)

# In addition to vision-only mode, tts processor and vocos also needs to be initialized
model.init_tts()

## ***Test Inference***
- This is also code snippet introduced in huggingface for test
- Check this code first before main test

In [ ]:
import math
import numpy as np
from PIL import Image
from moviepy.editor import VideoFileClip
import tempfile
import librosa
import soundfile as sf

def get_video_chunk_content(video_path, flatten=True):
    video = VideoFileClip(video_path)
    print('video_duration:', video.duration)
    
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=True) as temp_audio_file:
        temp_audio_file_path = temp_audio_file.name
        video.audio.write_audiofile(temp_audio_file_path, codec="pcm_s16le", fps=16000)
        audio_np, sr = librosa.load(temp_audio_file_path, sr=16000, mono=True)
    num_units = math.ceil(video.duration)
    
    # 1 frame + 1s audio chunk
    contents= []
    for i in range(num_units):
        frame = video.get_frame(i+1)
        image = Image.fromarray((frame).astype(np.uint8))
        audio = audio_np[sr*i:sr*(i+1)]
        if flatten:
            contents.extend(["<unit>", image, audio])
        else:
            contents.append(["<unit>", image, audio])
    
    return contents

video_path="/home/ubuntu/test4.mp4"
# if use voice clone prompt, please set ref_audio
# ref_audio_path = 'assets/demo.wav'
# ref_audio, _ = librosa.load(ref_audio_path, sr=16000, mono=True)
sys_msg = model.get_sys_prompt(mode='omni', language='en')
# or use default prompt
# sys_msg = model.get_sys_prompt(mode='omni', language='en')

contents = get_video_chunk_content(video_path)
msg = {"role":"user", "content": contents}
msgs = [sys_msg, msg]

# please set generate_audio=True and output_audio_path to save the tts result
generate_audio = True
output_audio_path = 'output.wav'

res = model.chat(
    msgs=msgs,
    tokenizer=tokenizer,
    sampling=True,
    temperature=0.5,
    max_new_tokens=4096,
    omni_input=True, # please set omni_input=True when omni inference
    use_tts_template=True,
    generate_audio=generate_audio,
    output_audio_path=output_audio_path,
    max_slice_nums=1,
    use_image_id=False,
    return_dict=True
)
print(res)

## You will get the answer: The person in the picture is skiing down a snowy slope.
# import IPython
# IPython.display.Audio('output.wav')


## ***Main task***
- Define video paths and corresponding truth label first

In [ ]:
video_paths = ["test.mp4", "test2.mp4", "test3.mp4", "test4.mp4", "test5.mp4", "test6.mp4"]
truth_labels = ["[[Abnormal]]", "[[Abnormal]]", "[[Normal]]", "[[Normal]]", "[[Normal]]", "[[Normal]]" ]

- Repeat inference-correction loop until all inferences are fully correct

In [ ]:
import math
import numpy as np
from PIL import Image
from moviepy.editor import VideoFileClip
import tempfile
import librosa
import ollama
import re

##############################
# Extract video/audio chunks #
##############################
def get_video_chunk_content(video_path, flatten=True):
    video = VideoFileClip(video_path)
    print('video_duration:', video.duration)
    
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=True) as temp_audio_file:
        temp_audio_file_path = temp_audio_file.name
        video.audio.write_audiofile(temp_audio_file_path, codec="pcm_s16le", fps=16000)
        audio_np, sr = librosa.load(temp_audio_file_path, sr=16000, mono=True)

    num_units = math.ceil(video.duration)
    contents = []
    for i in range(num_units):
        frame = video.get_frame(i+1)
        image = Image.fromarray((frame).astype(np.uint8))
        audio = audio_np[sr*i:sr*(i+1)]
        if flatten:
            contents.extend(["<unit>", image, audio])
        else:
            contents.append(["<unit>", image, audio])
    return contents

#####################################
# MiniCPM inference with confidence #
#####################################
def run_minicpm(video_path, prompt, model, tokenizer):
    contents = get_video_chunk_content(video_path)
    contents.append("<unit>")
    contents.append(prompt)
    
    sys_msg = model.get_sys_prompt(mode='omni', language='en')
    msg = {"role": "user", "content": contents}
    msgs = [sys_msg, msg]
    
    res = model.chat(
        msgs=msgs,
        tokenizer=tokenizer,
        sampling=True,
        temperature=1.0,
        max_new_tokens=4096,
        omni_input=True,
        use_tts_template=False,
        generate_audio=False,
        max_slice_nums=1,
        use_image_id=False,
        return_dict=True
    )
    
    output_text = res["text"].strip()
    
    # Regex to extract [[Normal|Abnormal]]
    match_pred = re.search(r"\[\[(Normal|Abnormal)\]\]", output_text)
    pred_text = match_pred.group(0) if match_pred else output_text
    
    # Regex to extract confidence (float number)
    match_conf = re.search(r"confidence[: ]*([0-9]*\.?[0-9]+)", output_text)
    confidence = float(match_conf.group(1)) if match_conf else 0.0
    
    return pred_text, confidence, output_text

#################################
# Llama3.1:8B prompt correction #
#################################
def correct_prompt_with_llama(old_prompt, mini_pred, confidence):
    instruction = f"""
You are optimizing a prompt for a multimodal LLM to detect scam videos.
MiniCPM received the following prompt but made an incorrect or low-confidence prediction.

Original prompt:
\"\"\"{old_prompt}\"\"\"

MiniCPM's output: {mini_pred} (confidence: {confidence})

The video clearly advertises high-paying jobs using stacked products and assembly tricks, which is a scam. 
Revise the prompt so that MiniCPM will correctly classify such videos as scams with high confidence, always ending the answer with [[Abnormal]] (confidence: x.x(close to 1.0)).
Or else, if it is not scam and normal then answer with [[Normal]] (confidence: x.x(close to 1.0))
Make the prompt direct, concise, and emphasize that these videos are scams and must be classified [[Abnormal]] (confidence: x.x(close to 1.0)).
"""
    llama_response = ollama.chat(
        model="llama3.1:8b",
        messages=[{'role': 'user', 'content': instruction}],
        options = {'temperature': 1.0}
    )
    content = llama_response['message']['content']
    return content.strip()

####################################
# Repeat inference-correction loop #
####################################
def auto_loop(video_path, initial_prompt, truth_label, model, tokenizer, max_iter=5, confidence_thresh=0.9):
    prompt = initial_prompt
    for i in range(max_iter):
        pred, conf, output = run_minicpm(video_path, prompt, model, tokenizer)
        print(f"[MiniCPM] iteration {i+1} output: {output}")
        print(f"[MiniCPM] iteration {i+1} summary: {pred} ★confidence: {conf}★")
        
        if pred == truth_label and conf >= confidence_thresh:
            print("[OK] Prediction matches truth with high confidence.")
            break
        else:
            print("[!] Prediction incorrect or low confidence, correcting prompt...")
            prompt = correct_prompt_with_llama(prompt, pred, conf)
            print(f"[!] New optimized prompt:\n{prompt}\n")
    return pred, prompt

while True:
    is_fully_correct = True
    
    for video_path, truth_label in zip(video_paths, truth_labels):
        
        initial_prompt = """This YouTube video shows people stacking and assembling products while advertising high-paying jobs. Most videos of this kind are scams. Evaluate if this video is a scam. If high probability, answer [[Abnormal]]; if low probability, answer [[Normal]]; Your answer must always end with [[Normal]] or [[Abnormal]]. with (Confidence: x.x)"""

        final_pred, final_prompt = auto_loop(
            video_path,
            initial_prompt,
            truth_label,
            model,
            tokenizer,
            max_iter=5,
            confidence_thresh=0.9
        )   

        if final_pred != truth_label:
            is_fully_correct = False
            break

        print(f"Final Prediction: {final_pred}")
        print(f"Final Optimized Prompt:\n{final_prompt}")
    
    if is_fully_correct:
        print(f"Perfect Optimized Prompt:\n{final_prompt}")
        break
